<a href="https://colab.research.google.com/github/olaidczak/dl-project-1/blob/main/dl_projekt_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
!pip uninstall -y sympy
!pip install sympy==1.13.3
!pip install timm -q
import timm
from torch import nn
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models
import torch.optim as optim
import requests
import random
import kagglehub
import datetime
import os
import json
import numpy as np
from datetime import datetime
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from timeit import default_timer as timer


Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 63.5 MB/s eta 0:00:00


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 134
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# deterministic gpu results
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device


'cuda'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
image_path = Path(kagglehub.dataset_download("mengcius/cinic10"))
train_dir = image_path / "train"
valid_dir = image_path / "valid"
test_dir = image_path / "test"

Using Colab cache for faster access to the 'cinic10' dataset.


In [ ]:
#  CINIC-10 statistics
CINIC_MEAN = [0.47889522, 0.47227842, 0.43047404]
CINIC_STD  = [0.24205776, 0.23828046, 0.25874835]

from torchvision.transforms import v2

cutmix = v2.CutMix(num_classes=10, alpha=1.0)


# ── Fabryka transformów ───────────────────────────────────────────────────────
def build_transforms(data_aug: str):
    """

    data_aug:
        'none'     — only  Normalize
        'standard' — RandomCrop + HorizontalFlip + ColorJitter + Normalize
        'cutmix'   — standard + CutMix
    """
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
    ])

    if data_aug == 'none':
        train_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
        ])
    elif data_aug in ('standard', 'cutmix'):
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
            transforms.ToTensor(),
            transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
        ])
    else:
        raise ValueError(f"Nieznana augmentacja: '{data_aug}'. Użyj: 'none', 'standard', 'cutmix'")

    return train_transform, eval_transform


In [ ]:
def build_fourblocklayer(num_classes=10, dropout=0):
    # tiny four block layer cnn
    # every layer gets the same dropout(modification from the original architecture)
    rates = [dropout] * 4

    class FourBlockCNN(nn.Module):
        def __init__(self, num_classes, rates):
            super(FourBlockCNN, self).__init__()
            self.block1 = self._make_block(3, 32, rates[0])
            self.block2 = self._make_block(32, 64, rates[1])
            self.block3 = self._make_block(64, 128, rates[2])
            self.block4 = self._make_block(128, 256, rates[3])

            self.classifier = nn.Sequential(
                nn.Conv2d(256, 256, kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(256, num_classes, kernel_size=1),
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Softmax(1)
            )

        def _make_block(self, in_channels, out_channels, p):
            layers = []
            for i in range(4):
                stride = 2 if i == 3 else 1
                curr_in = in_channels if i == 0 else out_channels
                layers.append(nn.Conv2d(curr_in, out_channels, kernel_size=3, stride=stride, padding=1))
                layers.append(nn.BatchNorm2d(out_channels))
                layers.append(nn.ReLU(inplace=True))

            if p > 0:
                layers.append(nn.Dropout(p))
            return nn.Sequential(*layers)

        def forward(self, x):
            x = self.block1(x)
            x = self.block2(x)
            x = self.block3(x)
            x = self.block4(x)
            return self.classifier(x)

    return FourBlockCNN(num_classes, rates)

In [ ]:
def build_resnet18(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
  model = models.resnet18(weights=weights)
  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
  model.maxpool = nn.Identity()

  # Add dropout
  model.fc = nn.Sequential(
    nn.Dropout(dropout),
    nn.Linear(model.fc.in_features, num_classes)
)
  return model

In [ ]:
def build_densenet121(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
    model = models.densenet121(weights=weights)

    # Adjust for 32x32 images
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()

    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(num_ftrs, num_classes)
    )
    return model


def build_ghostnetv3(num_classes=10, dropout=0):

    backbone = timm.create_model('ghostnetv3_100', pretrained=True, num_classes=0)

    old_conv = backbone.conv_stem
    backbone.conv_stem = nn.Conv2d(
        old_conv.in_channels, old_conv.out_channels,
        kernel_size=old_conv.kernel_size, stride=1,
        padding=old_conv.padding, bias=False
    )
    backbone.conv_stem.weight = old_conv.weight


    first_block = backbone.blocks[0][0]
    if hasattr(first_block, 'conv_dw'):
        first_block.conv_dw.stride = (1, 1)

    num_ftrs = backbone.num_features

    backbone.head = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(num_ftrs, num_classes)
    )
    return backbone


In [ ]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               data_aug: str = 'standard'):
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        if data_aug == 'cutmix':
            X, y = cutmix(X, y)
            y_pred = model(X)
            log_probs = torch.nn.functional.log_softmax(y_pred, dim=1)
            loss = -(y * log_probs).sum(dim=1).mean()
        else:
            y_pred = model(X)
            loss = loss_fn(y_pred, y)

        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_hard = y.argmax(dim=1) if y.dim() == 2 else y  #  soft labels  for CutMix
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y_hard).sum().item() / len(y_pred)

    train_loss = train_loss / len(dataloader)
    train_acc  = train_acc  / len(dataloader)
    return train_loss, train_acc


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

    test_loss = test_loss / len(dataloader)
    test_acc  = test_acc  / len(dataloader)
    return test_loss, test_acc


In [ ]:
import copy
import csv
import os

def save_checkpoint(epoch, model, optimizer, scheduler, results,
                    best_acc, best_weights, path,
                    curves_path=None, timestamp=None):

    checkpoint = {
        "epoch":              epoch,
        "model_state":        model.state_dict(),
        "optimizer_state":    optimizer.state_dict(),
        "scheduler_state":    scheduler.state_dict() if scheduler is not None else None,
        "results":            results,
        "best_acc_test":      best_acc,
        "best_model_weights": best_weights,
    }
    torch.save(checkpoint, path)

    if curves_path is not None and timestamp is not None:
        curves_header = ["timestamp", "epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
        curves_exists = os.path.isfile(curves_path)
        already_saved = 0
        if curves_exists:
            with open(curves_path, "r") as f:
                already_saved = sum(1 for row in csv.DictReader(f)
                                    if str(row["timestamp"]) == str(timestamp))
        with open(curves_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=curves_header)
            if not curves_exists:
                writer.writeheader()
            for ep in range(already_saved, epoch + 1):
                writer.writerow({
                    "timestamp":  timestamp,
                    "epoch":      ep + 1,
                    "train_loss": round(results["train_loss"][ep], 6),
                    "train_acc":  round(results["train_acc"][ep],  6),
                    "val_loss":   round(results["test_loss"][ep],  6),
                    "val_acc":    round(results["test_acc"][ep],   6),
                })
        print(f"  [Checkpoint] actualized curves.csv (epoki {already_saved+1}-{epoch+1})")


def load_checkpoint(path, model, optimizer, scheduler):
    """
    loads checkpoint and restores model/optimizer/scheduler state.
    Returns: (start_epoch, results, best_acc, best_weights)
    """
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt["scheduler_state"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    print(f"  [Checkpoint] Wznowienie od epoki {ckpt['epoch']+2} "
          f"(ostatnia zapisana: {ckpt['epoch']+1})")
    return ckpt["epoch"] + 1, ckpt["results"], ckpt["best_acc_test"], ckpt["best_model_weights"]


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          save_dir: str,
          scheduler=None,
          data_aug: str = 'standard',
          checkpoint_every: int = 5,
          model_name: str = 'model',
          resume_from: str = None,
          curves_path: str = None,
          timestamp: int = None):
    """
    resume_from : path to .ckpt — resume interrupted training
                  None — train from scratch
    """
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    best_acc_test = 0
    best_model_weights = copy.deepcopy(model.state_dict())
    start_epoch = 0

    # resuming
    if resume_from is not None and os.path.isfile(resume_from):
        start_epoch, results, best_acc_test, best_model_weights = load_checkpoint(
            resume_from, model, optimizer, scheduler
        )
    elif resume_from is not None:
        print(f"  [Checkpoint] Nie znaleziono {resume_from}, trening od zera.")

    checkpoint_path = os.path.join(save_dir, f"checkpoint_{timestamp}.ckpt")

    for epoch in tqdm(range(start_epoch, epochs)):
        train_loss, train_acc = train_step(
            model=model, dataloader=train_dataloader,
            loss_fn=loss_fn, optimizer=optimizer, data_aug=data_aug
        )
        test_loss, test_acc = test_step(
            model=model, dataloader=test_dataloader, loss_fn=loss_fn
        )

        if scheduler is not None:
            scheduler.step()

        if test_acc > best_acc_test:
            best_acc_test = test_acc
            best_model_weights = copy.deepcopy(model.state_dict())

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"valid_loss: {test_loss:.4f} | "
            f"valid_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item()   if isinstance(train_acc,  torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item()   if isinstance(test_loss,  torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item()    if isinstance(test_acc,   torch.Tensor) else test_acc)

        if (epoch + 1) % checkpoint_every == 0:
            save_checkpoint(
                epoch, model, optimizer, scheduler,
                results, best_acc_test, best_model_weights,
                checkpoint_path,
                curves_path=curves_path,
                timestamp=timestamp,
            )

    model.load_state_dict(best_model_weights)
    final_model_path = os.path.join(save_dir, f"{model_name}_{timestamp}.pt")
    torch.save(best_model_weights, final_model_path)
    print(f"  [Trening zakończony] Najlepszy model -> {final_model_path}")

    if os.path.isfile(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"  [Checkpoint] Usunięto tymczasowy checkpoint.")

    results["best_val_acc"] = best_acc_test
    return results


def save_results_to_csv(results, config, curves_path, experiments_path):

    timestamp  = config["timestamp"]
    num_epochs = len(results["train_loss"])

    curves_header = ["timestamp", "epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
    curves_exists = os.path.isfile(curves_path)
    already_saved = 0
    if curves_exists:
        with open(curves_path, "r") as f:
            already_saved = sum(1 for row in csv.DictReader(f)
                                if str(row["timestamp"]) == str(timestamp))
    with open(curves_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=curves_header)
        if not curves_exists:
            writer.writeheader()
        for epoch in range(already_saved, num_epochs):
            writer.writerow({
                "timestamp":  timestamp,
                "epoch":      epoch + 1,
                "train_loss": round(results["train_loss"][epoch], 6),
                "train_acc":  round(results["train_acc"][epoch],  6),
                "val_loss":   round(results["test_loss"][epoch],  6),
                "val_acc":    round(results["test_acc"][epoch],   6),
            })

    exp_header = [
        "timestamp", "model", "pretrained_weights", "optimizer",
        "learning_rate", "batch_size", "num_epochs", "weight_decay",
        "dropout", "scheduler", "data_augmentation",
        "best_val_acc", "training_time_s"
    ]
    exp_exists = os.path.isfile(experiments_path)
    with open(experiments_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=exp_header)
        if not exp_exists:
            writer.writeheader()
        writer.writerow({
            "timestamp":          timestamp,
            "model":              config["model"],
            "pretrained_weights": config["pretrained_weights"],
            "optimizer":          config["optimizer"],
            "learning_rate":      config["learning_rate"],
            "batch_size":         config["batch_size"],
            "num_epochs":         num_epochs,
            "weight_decay":       config["weight_decay"],
            "dropout":            config["dropout"],
            "scheduler":          config["scheduler"],
            "data_augmentation":  config["data_augmentation"],
            "best_val_acc":       round(results["best_val_acc"], 6),
            "training_time_s":    round(config["training_time"], 2),
        })

    print(f"saved curves  -> {curves_path}")
    print(f"saved experiments  -> {experiments_path}")


In [ ]:

#   'none'        — only  Normalize
#   'standard'    — RandomCrop + HorizontalFlip + Normalize
#   'cutmix'      — standard + CutMix
DATA_AUG    = 'standard'  # 'none' / 'standard' / 'cutmix'
BATCH_SIZE  = 32
NUM_WORKERS = 2

train_transform, eval_transform = build_transforms(DATA_AUG)

train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)
valid_data = datasets.ImageFolder(root=valid_dir, transform=eval_transform)
test_data  = datasets.ImageFolder(root=test_dir,  transform=eval_transform)

class_indices_train = defaultdict(list)
class_indices_valid = defaultdict(list)

for i, label in enumerate(train_data.targets):
    class_indices_train[label].append(i)

for i, label in enumerate(valid_data.targets):
    class_indices_valid[label].append(i)

selected_train, selected_valid = [], []
for label, inds in class_indices_train.items():
    random.shuffle(inds)
    selected_train.extend(inds[:int(0.75 * len(inds))])

for label, inds in class_indices_valid.items():
    random.shuffle(inds)
    selected_valid.extend(inds[:int(0.75 * len(inds))])

subset_train = Subset(train_data, selected_train)
subset_valid = Subset(valid_data, selected_valid)

train_dataloader = DataLoader(subset_train, batch_size=BATCH_SIZE,
                              num_workers=NUM_WORKERS, shuffle=True)
valid_dataloader = DataLoader(subset_valid, batch_size=BATCH_SIZE,
                              num_workers=NUM_WORKERS, shuffle=False)

print(f"Augmentation: {DATA_AUG} | Batch: {BATCH_SIZE} | "
      f"Train samples: {len(subset_train)} | Valid samples: {len(subset_valid)}")


Augmentacja: standard | Batch: 32 | Train samples: 67500 | Valid samples: 67500


In [ ]:
# MODEL:   'resnet18' | 'densenet121' | 'ghostnetv3'
MODEL              = 'densenet121'
NUM_EPOCHS         = 15
LEARNING_RATE      = 0.01
WEIGHT_DECAY       = 1e-4
DROPOUT            = 0
PRETRAINED_WEIGHTS = 'IMAGENET1K_V1'

SAVE_DIR = '/content/drive/MyDrive/data'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f'Utworzono folder: {SAVE_DIR}')

CURVES_CSV      = f'{SAVE_DIR}/curves.csv'
EXPERIMENTS_CSV = f'{SAVE_DIR}/experiments.csv'

RESUME_FROM = None
# RESUME_FROM = f'{SAVE_DIR}/checkpoint_{timestamp}.ckpt'

if MODEL == 'resnet18':
    model = build_resnet18(10, weights=PRETRAINED_WEIGHTS, dropout=DROPOUT).to(device)
elif MODEL == 'densenet121':
    model = build_densenet121(10, weights=PRETRAINED_WEIGHTS, dropout=DROPOUT).to(device)
elif MODEL == 'ghostnetv3':
    model = build_ghostnetv3(10, dropout=DROPOUT).to(device)
else:
    raise ValueError(f'Nieznany model: {MODEL}')

loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), momentum=0.9,
                      lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


scheduler      = None
SCHEDULER_NAME = 'None'

timestamp = int(datetime.now().timestamp())

# training
start_time = timer()
model_results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=valid_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS,
    save_dir=SAVE_DIR,
    scheduler=scheduler,
    data_aug=DATA_AUG,
    checkpoint_every=5,
    model_name=MODEL,
    resume_from=RESUME_FROM,
    curves_path=CURVES_CSV,
    timestamp=timestamp,
)
end_time = timer()
print(f'Total training time: {end_time - start_time:.3f} seconds')

config = {
    'timestamp':          timestamp,
    'model':              MODEL,
    'pretrained_weights': PRETRAINED_WEIGHTS,
    'optimizer':          'SGD',
    'learning_rate':      LEARNING_RATE,
    'batch_size':         BATCH_SIZE,
    'weight_decay':       WEIGHT_DECAY,
    'dropout':            DROPOUT,
    'scheduler':          SCHEDULER_NAME,
    'data_augmentation':  DATA_AUG,
    'training_time':      end_time - start_time,
}

save_results_to_csv(
    results=model_results,
    config=config,
    curves_path=CURVES_CSV,
    experiments_path=EXPERIMENTS_CSV
)


FEW-SHOT LEARNING


In [ ]:

#  divided 10 classes of  CINIC-10 on seen/unseen:
#   SEEN   (6 klas, trening backbone): airplane, automobile, bird, cat, deer, dog
#   UNSEEN (4 klasy, few-shot test):   frog, horse, ship, truck

CINIC10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

SEEN_CLASSES   = [0, 1, 2, 3, 4, 5]  # airplane, automobile, bird, cat, deer, dog
UNSEEN_CLASSES = [6, 7, 8, 9]        # frog, horse, ship, truck

print(f"Seen   ({len(SEEN_CLASSES)} classes): {[CINIC10_CLASSES[i] for i in SEEN_CLASSES]}")
print(f"Unseen ({len(UNSEEN_CLASSES)} classes): {[CINIC10_CLASSES[i] for i in UNSEEN_CLASSES]}")


def build_seen_only_loader(data, seen_classes, batch_size, num_workers=2, shuffle=True):
    """
    builds dataloader containing ONLY examples from seen_classes.
    Used for backbone training — model doesn't see unseen classes.
    """
    targets = np.array(data.targets)
    mask = np.isin(targets, seen_classes)
    indices = np.where(mask)[0]
    subset = Subset(data, indices)
    return DataLoader(subset, batch_size=batch_size,
                      num_workers=num_workers, shuffle=shuffle)


fs_train_transform, fs_eval_transform = build_transforms('standard')

fs_train_data = datasets.ImageFolder(root=train_dir, transform=fs_train_transform)
fs_test_data  = datasets.ImageFolder(root=test_dir,  transform=fs_eval_transform)

# DataLoader only for seen classes
FS_BATCH_SIZE = 32
fs_train_loader = build_seen_only_loader(fs_train_data, SEEN_CLASSES, FS_BATCH_SIZE)
fs_val_loader   = build_seen_only_loader(fs_train_data, SEEN_CLASSES, FS_BATCH_SIZE, shuffle=False)

print(f"\nTrain loader (seen only): {len(fs_train_loader.dataset)} examples")
print(f"unique classes in  train: {sorted(set(np.array(fs_train_data.targets)[list(fs_train_loader.dataset.indices)]))}")


Seen   (6 klas): ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog']
Unseen (4 klas): ['frog', 'horse', 'ship', 'truck']

Train loader (seen only): 54000 przykładów
Unikalne klasy w train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [ ]:
def get_backbone(model):
    """
    returns backkbone of a model (nn.Module) without the classification head, ready to produce embeddings.
    """
    if isinstance(model, models.ResNet):
        # ResNet: delete last fc layer, so just add Flatten
        backbone = nn.Sequential(*list(model.children())[:-1], nn.Flatten())
    elif isinstance(model, models.DenseNet):
        # denset delete classifier, but features already has AdaptiveAvgPool2d, so just add Flatten
        backbone = nn.Sequential(
            model.features,
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
    else:

        # classification head is a last element of model.head, so we take all but the last layer of model.head and add it to backbone
        backbone = nn.Sequential(
            *list(model.children())[:-1],
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
    return backbone.to(device)



def prototypical_episode(embeddings, labels, n_way, k_shot, n_query):
    """
    embeddings : (N_total, D)
    labels     : (N_total,)
    n_way      : number of classes in the episode
    k_shot     : number of examples per class in the support set
    n_query    : number of examples per class in the query set

    """
    classes = torch.unique(labels).tolist()
    selected_classes = random.sample(classes, n_way)

    support_embeddings = []
    query_embeddings   = []
    query_labels       = []

    for idx, cls in enumerate(selected_classes):
        cls_mask = (labels == cls).nonzero(as_tuple=True)[0]
        perm = cls_mask[torch.randperm(len(cls_mask))]
        support_idx = perm[:k_shot]
        query_idx   = perm[k_shot:k_shot + n_query]

        support_embeddings.append(embeddings[support_idx])
        query_embeddings.append(embeddings[query_idx])
        query_labels.extend([idx] * len(query_idx))

    # prototypes: average of support embeddings for each class
    prototypes = torch.stack(
        [s.mean(dim=0) for s in support_embeddings]
    )  # (n_way, D)

    query_emb = torch.cat(query_embeddings)   # (n_way * n_query, D)
    query_lbl = torch.tensor(query_labels)    # (n_way * n_query,)

    # euclidean distance from query embeddings to prototypes
    dists = torch.cdist(query_emb.unsqueeze(0), prototypes.unsqueeze(0)).squeeze(0)
    # (n_way * n_query, n_way)

    predictions = dists.argmin(dim=1)         # nclosest prototype → predicted class
    acc = (predictions == query_lbl).float().mean().item()
    return acc


def prototypical_eval(backbone, embeddings, labels,
                      n_way=10, k_shot=5, n_query=15, n_episodes=600):
    """
    returns: (mean_acc, std_acc, 95%_confidence_interval)
    """
    accs = []
    for _ in tqdm(range(n_episodes), desc=f"{n_way}-way {k_shot}-shot"):
        acc = prototypical_episode(embeddings, labels, n_way, k_shot, n_query)
        accs.append(acc)

    accs = np.array(accs)
    mean = accs.mean()
    std  = accs.std()
    ci95 = 1.96 * std / np.sqrt(n_episodes)   # 95% confidence interval
    return mean, std, ci95


print("Funkcje Prototypical Networks załadowane.")


Funkcje Prototypical Networks załadowane.


In [ ]:

# we use 6 classes → classification head with 6 outputs
# after training we cut off the head and use backbone as embedding extractor

FS_MODEL      = 'four_block'       # 'densenet121' lub 'ghostnetv3'
FS_EPOCHS     = 15
FS_LR         = 0.01
FS_WD         = 1e-4
FS_PRETRAINED = 'IMAGENET1K_V1'
SAVE_DIR = '/content/drive/MyDrive/data'

FS_SAVE_DIR   = SAVE_DIR
FS_FEWSHOT_CSV = f'{FS_SAVE_DIR}/true_fewshot_results.csv'

# Seed
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Model with 6 classes
N_SEEN = len(SEEN_CLASSES)
if FS_MODEL == 'resnet18':
    fs_model = build_resnet18(N_SEEN, weights=FS_PRETRAINED, dropout=0).to(device)
elif FS_MODEL == 'densenet121':
    fs_model = build_densenet121(N_SEEN, weights=FS_PRETRAINED, dropout=0).to(device)
elif FS_MODEL == 'ghostnetv3':
    fs_model = build_ghostnetv3(N_SEEN, dropout=0).to(device)
elif FS_MODEL =='four_block':
    fs_model = build_fourblocklayer(N_SEEN, dropout=0).to(device)

fs_optimizer = optim.SGD(fs_model.parameters(), momentum=0.9,
                         lr=FS_LR, weight_decay=FS_WD)
fs_loss_fn   = nn.CrossEntropyLoss()
fs_timestamp = int(datetime.now().timestamp())

# model has 6 outputs, but dataloader labels are 0-9 → we need to remap seen class labels to 0-5
seen_label_map = {orig: new for new, orig in enumerate(SEEN_CLASSES)}
print(f"Mapowanie etykiet seen classes: {seen_label_map}")

print(f"\nTrening backbone ({FS_MODEL}) na {N_SEEN} seen classes przez {FS_EPOCHS} epok...")
fs_results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
best_acc = 0
best_weights = copy.deepcopy(fs_model.state_dict())

for epoch in tqdm(range(FS_EPOCHS)):
    # Train step with remapping
    fs_model.train()
    t_loss, t_acc = 0, 0
    for X, y in fs_train_loader:
        X = X.to(device)
        y = torch.tensor([seen_label_map[lbl.item()] for lbl in y]).to(device)
        pred = fs_model(X)
        loss = fs_loss_fn(pred, y)
        t_loss += loss.item()
        fs_optimizer.zero_grad()
        loss.backward()
        fs_optimizer.step()
        t_acc += (pred.argmax(1) == y).float().mean().item()
    t_loss /= len(fs_train_loader)
    t_acc  /= len(fs_train_loader)

    # Val step with remapping
    fs_model.eval()
    v_loss, v_acc = 0, 0
    with torch.inference_mode():
        for X, y in fs_val_loader:
            X = X.to(device)
            y = torch.tensor([seen_label_map[lbl.item()] for lbl in y]).to(device)
            pred = fs_model(X)
            v_loss += fs_loss_fn(pred, y).item()
            v_acc  += (pred.argmax(1) == y).float().mean().item()
    v_loss /= len(fs_val_loader)
    v_acc  /= len(fs_val_loader)

    if v_acc > best_acc:
        best_acc = v_acc
        best_weights = copy.deepcopy(fs_model.state_dict())

    fs_results["train_loss"].append(t_loss)
    fs_results["train_acc"].append(t_acc)
    fs_results["test_loss"].append(v_loss)
    fs_results["test_acc"].append(v_acc)
    print(f"Epoch {epoch+1} | train_loss: {t_loss:.4f} | train_acc: {t_acc:.4f} "
          f"| val_loss: {v_loss:.4f} | val_acc: {v_acc:.4f}")

fs_model.load_state_dict(best_weights)
fs_backbone_path = f"{FS_SAVE_DIR}/fs_backbone_{FS_MODEL}_{fs_timestamp}.pt"
torch.save(best_weights, fs_backbone_path)
print(f"\nBackbone saved -> {fs_backbone_path}")
print(f"Best val acc on seen classes: {best_acc:.4f}")


Mapowanie etykiet seen classes: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}

Trening backbone (four_block) na 6 seen classes przez 15 epok...


  0%|          | 0/15 [00:00<?, ?it/s]

Epoch 1 | train_loss: 1.6405 | train_acc: 0.3728 | val_loss: 1.5889 | val_acc: 0.4320
Epoch 2 | train_loss: 1.5815 | train_acc: 0.4439 | val_loss: 1.5562 | val_acc: 0.4737
Epoch 3 | train_loss: 1.5560 | train_acc: 0.4725 | val_loss: 1.5389 | val_acc: 0.4929
Epoch 4 | train_loss: 1.5347 | train_acc: 0.4979 | val_loss: 1.5102 | val_acc: 0.5227
Epoch 5 | train_loss: 1.5141 | train_acc: 0.5198 | val_loss: 1.5193 | val_acc: 0.5157
Epoch 6 | train_loss: 1.4981 | train_acc: 0.5372 | val_loss: 1.4872 | val_acc: 0.5484
Epoch 7 | train_loss: 1.4829 | train_acc: 0.5530 | val_loss: 1.4581 | val_acc: 0.5784
Epoch 8 | train_loss: 1.4720 | train_acc: 0.5641 | val_loss: 1.4517 | val_acc: 0.5850
Epoch 9 | train_loss: 1.4562 | train_acc: 0.5812 | val_loss: 1.4509 | val_acc: 0.5847


KeyboardInterrupt: 

In [ ]:
FS_MODEL = 'ghostnetv3'
N_SEEN = 6 #  amount of seen classes during backbone training
PATH = '/content/drive/MyDrive/data/fs_backbone_ghostnetv3_1774699706.pt'

# 2. initiatlization of empty model
if FS_MODEL == 'resnet18':
    fs_model = build_resnet18(N_SEEN, weights=FS_PRETRAINED, dropout=0).to(device)
elif FS_MODEL == 'densenet121':
    fs_model = build_densenet121(N_SEEN, weights=FS_PRETRAINED, dropout=0).to(device)
elif FS_MODEL == 'ghostnetv3':
    fs_model = build_ghostnetv3(N_SEEN, dropout=0).to(device)
elif FS_MODEL =='four_block':
    fs_model = build_fourblocklayer(N_SEEN, dropout=0).to(device)

# 3. loading the weights
fs_model.load_state_dict(torch.load(PATH, map_location=device))
fs_model.eval()
print("Model loaded successfully!")


Model wczytany pomyślnie!


In [ ]:
# evaluation on unseen classes requires extracting embeddings for all test examples, but only for unseen classes.

def extract_embeddings_for_classes(backbone, data, class_indices, batch_size=128):
    """
    extracts embeddings only for specified classes.
    Returns (embeddings, labels) where labels are ORIGINAL class indices.
    """
    targets = np.array(data.targets)
    mask = np.isin(targets, class_indices)
    indices = np.where(mask)[0]
    subset = Subset(data, indices)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=2)

    backbone.eval()
    all_emb, all_lbl = [], []
    with torch.inference_mode():
        for X, y in loader:
            emb = backbone(X.to(device)).cpu()
            all_emb.append(emb)
            all_lbl.append(y)
    return torch.cat(all_emb), torch.cat(all_lbl)

# gets backbone without classification head
fs_backbone = get_backbone(fs_model)

# extract embeddings for unseen classes from test set
print("Ekstrakcja embeddingów dla unseen classes z test setu...")
unseen_embeddings, unseen_labels = extract_embeddings_for_classes(
    fs_backbone, fs_test_data, UNSEEN_CLASSES
)
print(f"Embeddingi: {unseen_embeddings.shape}")
print(f"Unikalne klasy: {[CINIC10_CLASSES[i] for i in sorted(unseen_labels.unique().tolist())]}")

# evaluation paramaters 4-way 5-shot, 600 epizodów
N_WAY_FS   = len(UNSEEN_CLASSES)  # 4-way ( equal to number of unseen classes)
K_SHOT_FS  = 5 # how many examples per class in support set
N_QUERY_FS = 15
N_EPISODES_FS = 600

mean_acc_fs, std_acc_fs, ci95_fs = prototypical_eval(
    fs_backbone, unseen_embeddings, unseen_labels,
    n_way=N_WAY_FS, k_shot=K_SHOT_FS,
    n_query=N_QUERY_FS, n_episodes=N_EPISODES_FS
)

print(f"\n{'='*60}")
print(f" FEW-SHOT — {N_WAY_FS}-way {K_SHOT_FS}-shot")
print(f"Model: {FS_MODEL} | Seen: {[CINIC10_CLASSES[i] for i in SEEN_CLASSES]}")
print(f"Unseen: {[CINIC10_CLASSES[i] for i in UNSEEN_CLASSES]}")
print(f"Accuracy: {mean_acc_fs*100:.2f}% ± {ci95_fs*100:.2f}% (95% CI)")
print(f"Std: {std_acc_fs*100:.2f}%")
print(f"Baseline (losowy): {100/N_WAY_FS:.1f}%")
print(f"{'='*60}")

# saving results to CSV
fewshot_true_header = [
    "timestamp", "model", "seen_classes", "unseen_classes",
    "n_way", "k_shot", "n_query", "n_episodes",
    "mean_acc", "std_acc", "ci95", "baseline_random"
]
fs_exists = os.path.isfile(FS_FEWSHOT_CSV)
with open(FS_FEWSHOT_CSV, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fewshot_true_header)
    if not fs_exists:
        writer.writeheader()
    writer.writerow({
        "timestamp":       fs_timestamp,
        "model":           FS_MODEL,
        "seen_classes":    str([CINIC10_CLASSES[i] for i in SEEN_CLASSES]),
        "unseen_classes":  str([CINIC10_CLASSES[i] for i in UNSEEN_CLASSES]),
        "n_way":           N_WAY_FS,
        "k_shot":          K_SHOT_FS,
        "n_query":         N_QUERY_FS,
        "n_episodes":      N_EPISODES_FS,
        "mean_acc":        round(mean_acc_fs, 6),
        "std_acc":         round(std_acc_fs, 6),
        "ci95":            round(ci95_fs, 6),
        "baseline_random": round(1/N_WAY_FS, 4),
    })
print(f"saved to -> {FS_FEWSHOT_CSV}")


Ekstrakcja embeddingów dla unseen classes z test setu...
Embeddingi: torch.Size([36000, 1280])
Unikalne klasy: ['frog', 'horse', 'ship', 'truck']


4-way 100-shot:   0%|          | 0/600 [00:00<?, ?it/s]


PRAWDZIWY FEW-SHOT — 4-way 100-shot
Model: ghostnetv3 | Seen: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog']
Unseen: ['frog', 'horse', 'ship', 'truck']
Accuracy: 73.86% ± 0.44% (95% CI)
Std: 5.54%
Baseline (losowy): 25.0%
Zapisano -> /content/drive/MyDrive/data/true_fewshot_results.csv


ENSEMBLING

In [1]:

NUM_WORKERS = 0
SAVE_DIR = '/content/drive/MyDrive/data'

class EnsembleModel:
    """
    """
    def __init__(self, models_list):
        self.models = models_list
        for model in self.models:
            model.eval()

    def predict(self, x, voting='soft'):
        """
        voting: 'soft' or 'hard'
        """
        with torch.inference_mode():
            if voting == 'soft':
                # summing probabilities (softmax) from all models
                total_probs = None
                for model in self.models:
                    logits = model(x)
                    probs = torch.softmax(logits, dim=1)
                    if total_probs is None:
                        total_probs = probs
                    else:
                        total_probs += probs

                # returns class with highest average probability across models
                return torch.argmax(total_probs, dim=1)

            elif voting == 'hard':
                predictions = []
                for model in self.models:
                    logits = model(x)
                    preds = torch.argmax(logits, dim=1)
                    predictions.append(preds)

                # majority vote (mode) across model predictions
                stacked_preds = torch.stack(predictions, dim=1)
                final_preds, _ = torch.mode(stacked_preds, dim=1)
                return final_preds

def evaluate_ensemble(ensemble, dataloader, voting='soft'):
    correct = 0
    total = 0

    print(f"evaluation of ensemble (voting='{voting}')...")
    for X, y in tqdm(dataloader):
        X, y = X.to(device), y.to(device)
        preds = ensemble.predict(X, voting=voting)
        correct += (preds == y).sum().item()
        total += y.size(0)

    acc = correct / total
    return acc


# 1.loading models to ensemble
model1 = build_resnet18(10, weights=None).to(device)
model1.load_state_dict(torch.load(f'{SAVE_DIR}/resnet18_1774639867.pt'))

model2 = build_densenet121(10, weights=None).to(device)
model2.load_state_dict(torch.load(f'{SAVE_DIR}/densenet121_1774263404.pt'))

model3 = build_fourblocklayer(10).to(device)
model3.load_state_dict(torch.load(f'{SAVE_DIR}/four_block_1774642487.pt'))
# 2. list for the models to give to ensemble
ensemble_list = [model1, model2]

# 3.  evaluation
my_ensemble = EnsembleModel(ensemble_list)
test_dataloader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False
)
soft_acc = evaluate_ensemble(my_ensemble, valid_dataloader, voting='soft')
hard_acc = evaluate_ensemble(my_ensemble, valid_dataloader, voting='hard')
print(f"\nResult Soft Voting: {soft_acc*100:.2f}%")
print(f"Result Hard Voting: {hard_acc*100:.2f}%")

NameError: name 'build_resnet18' is not defined